# Fiche 9 — Comparer des cultures incomparables, et juger un classement

| | |
|---|---|
| 🎯 **Objectif** | Construire un score relatif pour comparer des cultures, et évaluer un classement face à des références (hasard, classement fixe) |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiche 1 ; médiane |
| 🔗 **Dans le projet** | Étape 2 et API (`/recommend`), rapport métier (recommandations) |

> **L'idée clé.** On ne compare pas des tonnes de pommes de terre à des tonnes de soja. On compare chaque culture à **son propre rendement habituel**. Et un classement n'est utile que s'il bat une référence **exigeante** : pas seulement le hasard, mais aussi « toujours le même ordre ».

In [1]:
import numpy as np
import pandas as pd

from agrireco import io, prepare
from agrireco.viz import CROPS_FR

_, fao_dir = io.download()
fao, _ = prepare.build_fao_table(*prepare.load_fao_raw(fao_dir))
mediane = fao.groupby("crop")["yield_t_ha"].median().sort_values()
mediane.rename(index=CROPS_FR).round(1)

crop
Sorgho              1.2
Soja                1.6
Blé                 2.5
Maïs                2.6
Riz                 3.5
Banane plantain     8.4
Patate douce        8.4
Igname              8.7
Manioc             10.0
Pomme de terre     16.0
Name: yield_t_ha, dtype: float64

## ⚠️ Piège : classer par tonnes brutes
Si l'on classe les cultures par t/ha, quel pays verrait autre chose que les tubercules en tête ?

In [2]:
annee = fao[fao["year"] == 2010]
gagnant_brut = annee.loc[annee.groupby("country")["yield_t_ha"].idxmax(), "crop"]
print("culture « gagnante » en t/ha brutes, par pays (2010) :")
print(gagnant_brut.map(CROPS_FR).value_counts().head())

culture « gagnante » en t/ha brutes, par pays (2010) :
crop
Pomme de terre     81
Manioc             17
Patate douce        9
Igname              6
Banane plantain     3
Name: count, dtype: int64


Pomme de terre, manioc, patate douce : des cultures **lourdes** (pleines d'eau). Ce classement dit « ce qui pèse le
plus », pas « ce qui réussit le mieux ici ».

## 1. Le rendement relatif
On divise le rendement par le rendement habituel de la culture : 1 = comme d'habitude, 1,3 = 30 % au-dessus.

In [3]:
annee = annee.assign(relatif=annee["yield_t_ha"] / annee["crop"].map(mediane))
france = annee[annee["country"] == "France"].sort_values("relatif", ascending=False)
france.assign(culture=france["crop"].map(CROPS_FR))[["culture", "yield_t_ha", "relatif"]].round(2)

,culture,yield_t_ha,relatif
4910,Sorgho,5.51,4.41
4841,Maïs,8.83,3.42
4956,Blé,7.04,2.85
4864,Pomme de terre,42.16,2.64
4933,Soja,2.75,1.73
4887,Riz,5.04,1.46


En France, le sorgho, le maïs et le blé dépassent de loin leur rendement habituel ; la pomme de terre, première en
tonnes, n'arrive qu'en 4ᵉ position. Le classement brut cachait cette information.

## 2. Juger un classement : quelles références ?
Supposons un « modèle » qui classe les cultures. Pour chaque pays, regardons si sa culture n° 1 est bien la
meilleure **parmi les cultures réellement pratiquées**. On le compare à :
- **le hasard** : une culture tirée au sort parmi celles du pays ;
- **un classement fixe** : toujours le même ordre, appris sur d'autres pays, sans regarder les conditions.

In [4]:
pays = np.random.default_rng(0).permutation(annee["country"].unique())
appris, evalues = pays[: len(pays) // 2], pays[len(pays) // 2 :]   # deux moitiés de pays
a, e = annee[annee["country"].isin(appris)].copy(), annee[annee["country"].isin(evalues)].copy()
# Le rendement habituel vient des seuls pays « appris », comme en fiche 4 (sinon, fuite d'information)
mediane_a = a.groupby("crop")["yield_t_ha"].median()
a["relatif"] = a["yield_t_ha"] / a["crop"].map(mediane_a)
e["relatif"] = e["yield_t_ha"] / e["crop"].map(mediane_a)

**Le classement fixe** : pour chaque culture, son rang moyen (1 = meilleure) parmi les cultures d'un même pays
« appris ». Il ne regarde jamais les conditions.

In [5]:
ordre_fixe = a.assign(r=a.groupby("country")["relatif"].rank(ascending=False)).groupby("crop")["r"].mean()

**L'évaluation**, simplifiée à des **paires** : dans chaque pays « évalué », on tire deux cultures pratiquées.
Laquelle a le meilleur rendement relatif ? Au hasard, on a raison une fois sur deux.

In [6]:
rng = np.random.default_rng(1)
ok_fixe = []
for _, g in e.groupby("country"):
    if len(g) < 2:
        continue
    paire = g.sample(2, random_state=rng.integers(1_000_000))
    meilleure = paire.loc[paire["relatif"].idxmax(), "crop"]
    choix_fixe = min(paire["crop"], key=lambda c: ordre_fixe.get(c, 99))
    ok_fixe.append(choix_fixe == meilleure)
print(f"{len(ok_fixe)} pays | bonne réponse — hasard : 50 % | classement fixe : {np.mean(ok_fixe):.0%}")

59 pays | bonne réponse — hasard : 50 % | classement fixe : 63%


Un classement **qui ne regarde même pas les conditions** fait déjà mieux que le hasard. Un modèle qui annoncerait
fièrement « mieux que le hasard » pourrait donc ne rien apporter de plus qu'une liste fixe. La bonne question
est : **bat-il le classement fixe ?** — avec une marge d'incertitude : ici une soixantaine de pays seulement, quelques
pays de plus ou de moins font bouger le taux de plusieurs points.

Dans ton projet : quelle référence fixe vas-tu construire, et sur quels pays ?

### ✍️ À toi de jouer

Refais le calcul des paires en classant par **tonnes brutes** (au lieu du rendement relatif) : la « meilleure culture » de la paire et l'ordre fixe utilisent alors `yield_t_ha`. Que devient le taux du classement fixe ? Pourquoi ce chiffre est-il trompeur ?

In [7]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
ordre_brut = a.assign(r=a.groupby("country")["yield_t_ha"].rank(ascending=False)).groupby("crop")["r"].mean()
rng = np.random.default_rng(1)
ok = []
for _, g in e.groupby("country"):
    if len(g) >= 2:
        paire = g.sample(2, random_state=rng.integers(1_000_000))
        ok.append(min(paire["crop"], key=lambda c: ordre_brut.get(c, 99)) == paire.loc[paire["yield_t_ha"].idxmax(), "crop"])
print(f"{np.mean(ok):.0%}")
```

**Pourquoi ?** le taux grimpe très haut : « la pomme de terre pèse plus que le soja » est vrai partout, donc facile à deviner. Un score élevé sur une question triviale ne dit rien d'utile. Choisir **ce qu'on compare** (relatif plutôt que brut) et **contre quoi** (une référence exigeante) compte autant que le modèle lui-même.
</details>

## ✅ À retenir

1. Des tonnes de cultures différentes ne se comparent pas : on divise par le rendement habituel de chaque culture.
2. Rendement relatif 1,3 = 30 % au-dessus de ce qu'on obtient d'habitude pour cette culture.
3. Un classement s'évalue face à des références : le hasard (facile à battre) et un classement fixe (exigeant).
4. On n'évalue que parmi les cultures réellement pratiquées : une culture jamais plantée n'a pas de vérité terrain.
5. « Mieux que le hasard » ne suffit pas pour dire qu'un modèle est utile.